# Module 2 – Working with OpenAI Embedding Models
### Course: OpenAI Embeddings API | Pluralsight

In [ ]:
# !pip install openai numpy datasets tenacity tiktoken

In [ ]:
import os
import getpass
import json
import time
from openai import OpenAI
from tenacity import retry, wait_random_exponential, stop_after_attempt

## TODO: #1


---
## Clip 2: Generating Embeddings with the API

Let's explore the full API response and understand how to extract the embedding vector.

In [ ]:
sample_text = "This restaurant has a great atmosphere and delicious food."

## TODO: #2

In [ ]:
# EMBEDDING MULTIPLE TEXTS IN ONE API CALL
yelp_reviews_sample = [
    "Amazing ramen, the broth is rich and complex. Will definitely come back.",
    "Disappointing service, cold food, won't return.",
    "Best tacos I've ever had. Authentic flavors and generous portions.",
    "Nice ambiance but the pasta was overcooked and bland.",
    "The sushi is fresh and beautifully presented. Great sake selection too.",
]

# TODO: #3


---
## Clip 3: Batch Processing and Scaling Embeddings

In [ ]:
# TODO: #4
# Load Dataset

In [ ]:
#TODO: #5
# Estimate Token Usage and Cost


In [ ]:
#TODO: #6
# Make function retryable on errors

def get_embeddings_with_retry(
    texts: list[str],
    model: str = "text-embedding-3-small"
) -> list[list[float]]:
    """
    Embed a batch of texts with automatic retry on rate limit errors.
    tenacity retries with exponential backoff: 1s, 2s, 4s, 8s... up to 20s.
    """
    response = client.embeddings.create(input=texts, model=model)
    # response.data is a list of Embedding objects, sorted by index
    return [item.embedding for item in sorted(response.data, key=lambda x: x.index)]


# Quick test
test_vecs = get_embeddings_with_retry(["test review one", "test review two"])
print(f"Batch of 2: returned {len(test_vecs)} vectors, each with {len(test_vecs[0])} dims")

In [ ]:
BATCH_SIZE = 100
OUTPUT_FILE = "yelp_embeddings.json"

# Skip if already embedded
if os.path.exists(OUTPUT_FILE):
    print(f"Found existing file: {OUTPUT_FILE} — loading instead of re-embedding")
    with open(OUTPUT_FILE) as f:
        embedded_reviews = json.load(f)
else:
    start_time = time.time()
    embedded_reviews = []

    batches = [reviews[i:i + BATCH_SIZE] for i in range(0, len(reviews), BATCH_SIZE)]
    total_batches = len(batches)

    for batch_num, batch in enumerate(batches, 1):
        texts = [r["text"] for r in batch]
        embeddings = get_embeddings_with_retry(texts)

        for review, embedding in zip(batch, embeddings):
            embedded_reviews.append({**review, "embedding": embedding})

        if batch_num % 10 == 0 or batch_num == total_batches:
            elapsed = time.time() - start_time
            print(f"Batch {batch_num}/{total_batches} | {len(embedded_reviews)} reviews done | {elapsed:.1f}s elapsed")

    # Save to disk
    with open(OUTPUT_FILE, "w") as f:
        json.dump(embedded_reviews, f)

    elapsed = time.time() - start_time
    print(f"\nCompleted in {elapsed:.1f}s")
    print(f"Saved {len(embedded_reviews)} embedded reviews to {OUTPUT_FILE}")

print(f"\nReady: {len(embedded_reviews)} embedded reviews")
print(f"Vector size: {len(embedded_reviews[0]['embedding'])} dimensions")